## Welcome to your notebook.


#### Run this cell to connect to your GIS and get started:

In [30]:
from arcgis.gis import GIS
from arcgis.apps.survey123 import SurveyManager
import urllib.request, zipfile, io, os, shutil, subprocess, yaml
import pandas as pd

gis = GIS("home")
TITLE = "Whatcom Creek Salmon Survey"
REPO_ZIP = "https://github.com/MegaDeadCowboy/whatcom-creek-salmon-monitoring/archive/refs/heads/main.zip"
REPO = "/arcgis/home/repo"
SCHEMA = f"{REPO}/docs/schema.yaml"
XLSX = f"{REPO}/form/whatcom_salmon.xlsx"

You are logged on as carver.rasmussen.gis with an administrator role, proceed with caution.


#### Now you are ready to start!

In [31]:
os.chdir("/arcgis/home")
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(REPO_ZIP).read())).extractall("/arcgis/home")
os.rename("/arcgis/home/whatcom-creek-salmon-monitoring-main", REPO)
os.chdir(REPO)
schema = yaml.safe_load(open(SCHEMA))
print("schema version:", schema["version"])

schema version: 0.1.2


In [32]:
!python scripts/build_xlsform.py docs/schema.yaml form/whatcom_salmon.xlsx
survey = pd.read_excel(XLSX, sheet_name="survey")
choices = pd.read_excel(XLSX, sheet_name="choices")
thr = survey.query("name == 'location'")["body::accuracyThreshold"].iloc[0]
expected = schema.get("gps_accuracy_block_m", schema["gps_accuracy_warn_m"])
assert thr == expected, f"threshold {thr} != schema {expected} (stale zip?)"
print("accuracyThreshold:", thr)
choices.query("list_name == 'reach'")

Wrote form/whatcom_salmon.xlsx (15 fields, version 2026100221)
accuracyThreshold: 50.0


,list_name,name,label
0,reach,R1,R1: Mouth to Maritime Heritage Park
1,reach,R2,R2: Maritime Heritage Park to Woburn St
2,reach,R3,R3: Woburn St to WWU Fisheries Rental
3,reach,R4,R4: WWU Rental to Lake Outlet


In [33]:
hits = gis.content.search(f'title:"{TITLE}"', max_items=50)
forms = [i for i in hits if i.type == "Form" and i.title == TITLE]
svcs = [i for i in hits if i.type == "Feature Service" and i.title == TITLE
        and "View Service" not in i.typeKeywords]
assert len(forms) == 1 and len(svcs) == 1, f"expected 1 form + 1 layer, got {forms} {svcs}"
s = SurveyManager(gis).get(forms[0].id)
lyr = svcs[0].layers[0]
print(forms[0].homepage, svcs[0].homepage, sep="\n")

https://www.arcgis.com/home/item.html?id=86454697680343f1b09807d79dd9216e
https://www.arcgis.com/home/item.html?id=1d616f8cf70d4384ade27ee41b53de19


In [34]:
ftype = {"decimal": "esriFieldTypeDouble", "text": "esriFieldTypeString"}
existing = {f["name"] for f in lyr.properties.fields}
new = []
for sf in schema["fields"]:
    if sf.get("system") and sf["name"] not in existing:
        d = {"name": sf["name"], "alias": sf["label"], "type": ftype[sf["type"]],
             "nullable": True, "editable": True}
        if sf["type"] == "text":
            d["length"] = sf.get("max_length", 255)
        new.append(d)
print("adding:", [d["name"] for d in new] or "nothing")
if new:
    print(lyr.manager.add_to_definition({"fields": new}))

updates = []
for f in lyr.properties.fields:
    sf = next((x for x in schema["fields"] if x["name"] == f["name"] and x.get("domain")), None)
    if sf and f.get("domain"):
        d = dict(f["domain"])
        d["codedValues"] = [{"name": v, "code": k} for k, v in schema["domains"][sf["domain"]].items()]
        updates.append({"name": f["name"], "domain": d})
print(lyr.manager.update_definition({"fields": updates}))

adding: nothing
{'success': True}


In [35]:
s.publish(xlsform=XLSX)

/opt/conda/lib/python3.13/site-packages/arcgis/apps/survey123/_publish_functions.py:1954: UserWarning: Warning: ["[row : 2] Question has no label: {'type': 'username', 'name': 'username'}"]
  warnings.warn(f'Warning: {str(response_json["warnings"])}')


<Survey @ Whatcom Creek Salmon Survey>

In [36]:
lyr = svcs[0].layers[0]
names = [f["name"] for f in lyr.properties.fields]
missing = [f["name"] for f in schema["fields"]
           if f["type"] not in ("geopoint", "image") and f["name"] not in names]
print("missing fields:", missing or "none")
reach = next(f for f in lyr.properties.fields if f["name"] == "reach")["domain"]["codedValues"]
print([c["name"] for c in reach])

missing fields: none
['R1: Mouth to Maritime Heritage Park', 'R2: Maritime Heritage Park to Woburn St', 'R3: Woburn St to WWU Fisheries Rental', 'R4: WWU Rental to Lake Outlet']


In [37]:
# s = SurveyManager(gis).create(title=TITLE, tags="salmon,Survey123,Whatcom Creek")